# Experiment 3B: confirmatory coupled-regime comparison

The primary comparison is HCR against Static under the regime selected in the exploratory study. Other methods are secondary comparisons. The confirmatory protocol fixes priors `{0.55, 0.60}`, abstention costs `{0.20, 0.25}`, three latent source-quality regimes, seed and 100 final test instances with IDs 20000–20099. Protocol changes after observing final outcomes would define a new analysis.


## Environment and execution

Dependencies: Python 3.10+, NumPy, pandas, SciPy and Jupyter; matplotlib is used for optional figures. Diagnostic and final profiles are configured in the execution cell. `RESUME=True` preserves checkpoints for the same settings. Outputs are written under `experiment3b_confirmatory_outputs`, separately from exploratory results.


In [ ]:

from __future__ import annotations

import itertools
from dataclasses import dataclass
from functools import lru_cache
from typing import Dict, Iterable, List, Sequence, Tuple

import numpy as np
from scipy.optimize import linprog


Array = np.ndarray


@dataclass(frozen=True)
class SourceSpec:
    """Nominal binary source specification."""
    alpha: float  # P(E=1 | Y=1)
    beta: float   # P(E=1 | Y=0)
    cost: float
    name: str


def make_channel(alpha: float, beta: float) -> Array:
    """
    Return a binary evidence channel with rows Y={0,1} and columns E={0,1}.
    """
    if not (0.0 <= alpha <= 1.0 and 0.0 <= beta <= 1.0):
        raise ValueError("alpha and beta must lie in [0, 1].")
    return np.array(
        [
            [1.0 - beta, beta],
            [1.0 - alpha, alpha],
        ],
        dtype=float,
    )


def _deduplicate_channels(channels: Array, names: Sequence[str]) -> Tuple[Array, List[str]]:
    seen = set()
    kept_channels: List[Array] = []
    kept_names: List[str] = []

    for channel, name in zip(channels, names):
        key = tuple(np.round(channel.ravel(), 12))
        if key not in seen:
            seen.add(key)
            kept_channels.append(channel)
            kept_names.append(name)

    return np.stack(kept_channels), kept_names


def build_coupled_scenarios(
    sources: Sequence[SourceSpec],
    rho: float,
    eps: float = 1e-9,
) -> Tuple[Array, List[List[Tuple[float, float]]], Array, List[str]]:
    """
    Build a finite, coupled ambiguity set.

    The latent regimes shift all sources coherently. This deliberately creates
    dependence across source models so that the history-consistent and fully
    rectangular formulations need not coincide.

    Returns
    -------
    scenario_channels:
        Shape (n_scenarios, n_sources, 2, 2).
    source_projection_points:
        Per-source projected (alpha, beta) channel points for Rect-ROSE.
    nominal_channels:
        Shape (n_sources, 2, 2).
    scenario_names:
        Names of retained coupled regimes.
    """
    if rho < 0:
        raise ValueError("rho must be nonnegative.")

    regimes = [
        (0.0, 0.0, "nominal"),
        (-rho, +rho, "pessimistic"),
        (+rho, -rho, "optimistic"),
        (-rho, -rho, "low-positive-rate"),
        (+rho, +rho, "high-positive-rate"),
    ]

    raw_scenarios: List[Array] = []
    names: List[str] = []

    for delta_alpha, delta_beta, regime_name in regimes:
        scenario = []
        for source in sources:
            alpha = float(np.clip(source.alpha + delta_alpha, eps, 1.0 - eps))
            beta = float(np.clip(source.beta + delta_beta, eps, 1.0 - eps))
            scenario.append(make_channel(alpha, beta))
        raw_scenarios.append(np.stack(scenario))
        names.append(regime_name)

    scenario_channels, names = _deduplicate_channels(np.stack(raw_scenarios), names)

    source_projection_points: List[List[Tuple[float, float]]] = []
    for source_index in range(len(sources)):
        points: List[Tuple[float, float]] = []
        for scenario_index in range(scenario_channels.shape[0]):
            channel = scenario_channels[scenario_index, source_index]
            point = (float(channel[1, 1]), float(channel[0, 1]))
            if not any(np.allclose(point, existing) for existing in points):
                points.append(point)
        source_projection_points.append(points)

    nominal_channels = np.stack(
        [make_channel(source.alpha, source.beta) for source in sources]
    )

    return scenario_channels, source_projection_points, nominal_channels, names


def stop_value_precise(p_y1: float, abstention_cost: float) -> Tuple[float, str]:
    risks = [p_y1, 1.0 - p_y1, abstention_cost]
    actions = ["answer_0", "answer_1", "abstain"]
    index = int(np.argmin(risks))
    return float(risks[index]), actions[index]


def nominal_dp(
    prior_y1: float,
    channels: Array,
    costs: Array,
    abstention_cost: float,
) -> Tuple[Tuple[float, str, Dict[int, float]], Dict]:
    """Exact nominal dynamic program for binary sources."""

    n_sources = len(costs)
    memo: Dict = {}

    def recurse(omega: Array, remaining: Tuple[int, ...]):
        key = (tuple(float(value) for value in omega), remaining)
        if key in memo:
            return memo[key]

        total = float(omega.sum())
        p_y1 = float(omega[1] / total)
        stopping_value, stopping_action = stop_value_precise(
            p_y1, abstention_cost
        )

        best_value = stopping_value
        best_action = stopping_action
        continuation_values: Dict[int, float] = {}

        for source_index in remaining:
            value = float(costs[source_index])
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )

            for evidence in (0, 1):
                next_omega = omega * channels[source_index, :, evidence]
                evidence_probability = float(next_omega.sum() / total)

                if evidence_probability > 0.0:
                    child_value = recurse(next_omega, next_remaining)[0]
                    value += evidence_probability * child_value

            continuation_values[source_index] = value

            if value < best_value - 1e-12:
                best_value = value
                best_action = f"query_{source_index}"

        memo[key] = (best_value, best_action, continuation_values)
        return memo[key]

    root = recurse(
        np.array([1.0 - prior_y1, prior_y1], dtype=float),
        tuple(range(n_sources)),
    )
    return root, memo


def nominal_dp_from_state(
    omega: Array,
    channels: Array,
    costs: Array,
    abstention_cost: float,
    remaining: Tuple[int, ...],
) -> float:
    """Nominal continuation value from an unnormalized binary belief."""

    memo: Dict = {}

    def recurse(current_omega: Array, current_remaining: Tuple[int, ...]) -> float:
        key = (tuple(float(value) for value in current_omega), current_remaining)
        if key in memo:
            return memo[key]

        total = float(current_omega.sum())
        p_y1 = float(current_omega[1] / total)
        best_value, _ = stop_value_precise(p_y1, abstention_cost)

        for source_index in current_remaining:
            value = float(costs[source_index])
            next_remaining = tuple(
                index
                for index in current_remaining
                if index != source_index
            )

            for evidence in (0, 1):
                next_omega = (
                    current_omega * channels[source_index, :, evidence]
                )
                evidence_probability = float(next_omega.sum() / total)

                if evidence_probability > 0.0:
                    value += evidence_probability * recurse(
                        next_omega, next_remaining
                    )

            best_value = min(best_value, value)

        memo[key] = float(best_value)
        return memo[key]

    return recurse(np.asarray(omega, dtype=float), remaining)


def hcr_dp(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
) -> Tuple[Tuple[float, str, Dict[int, float], Tuple[float, float, float]], Dict]:
    """
    History-Consistent Rectangular dynamic program over a finite scenario set.
    """

    n_scenarios, n_sources, _, _ = scenario_channels.shape
    memo: Dict = {}

    def likelihood_matrix(history: Tuple[Tuple[int, int], ...]) -> Array:
        likelihood = np.tile(
            np.array([1.0 - prior_y1, prior_y1], dtype=float),
            (n_scenarios, 1),
        )

        for source_index, evidence in history:
            likelihood *= scenario_channels[:, source_index, :, evidence]

        return likelihood

    def recurse(
        history: Tuple[Tuple[int, int], ...],
        remaining: Tuple[int, ...],
    ):
        key = (history, remaining)
        if key in memo:
            return memo[key]

        likelihood = likelihood_matrix(history)
        reach_probability = likelihood.sum(axis=1)
        active = reach_probability > 0.0

        if not np.any(active):
            # Every complete model assigns this history zero probability.
            memo[key] = (0.0, "abstain", {}, (0.0, 0.0, float(abstention_cost)))
            return memo[key]

        posterior_y1 = likelihood[active, 1] / reach_probability[active]

        answer_0_risk = float(np.max(posterior_y1))
        answer_1_risk = float(np.max(1.0 - posterior_y1))
        risks = [answer_0_risk, answer_1_risk, abstention_cost]
        actions = ["answer_0", "answer_1", "abstain"]

        stopping_value = float(min(risks))
        stopping_action = actions[int(np.argmin(risks))]

        best_value = stopping_value
        best_action = stopping_action
        continuation_values: Dict[int, float] = {}

        for source_index in remaining:
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )

            child_values = {
                evidence: recurse(
                    history + ((source_index, evidence),),
                    next_remaining,
                )[0]
                for evidence in (0, 1)
            }

            scenario_expectations = np.zeros(n_scenarios, dtype=float)

            for evidence in (0, 1):
                numerator = (
                    likelihood
                    * scenario_channels[:, source_index, :, evidence]
                ).sum(axis=1)

                evidence_probability = np.divide(
                    numerator,
                    reach_probability,
                    out=np.zeros_like(numerator),
                    where=active,
                )

                scenario_expectations += (
                    evidence_probability * child_values[evidence]
                )

            continuation_value = float(
                costs[source_index]
                + np.max(scenario_expectations[active])
            )

            continuation_values[source_index] = continuation_value

            if continuation_value < best_value - 1e-12:
                best_value = continuation_value
                best_action = f"query_{source_index}"

        memo[key] = (
            best_value,
            best_action,
            continuation_values,
            (answer_0_risk, answer_1_risk, abstention_cost),
        )
        return memo[key]

    root = recurse(tuple(), tuple(range(n_sources)))
    return root, memo


def _deduplicate_beliefs(values: Iterable[float]) -> Tuple[float, ...]:
    return tuple(sorted(set(round(float(value), 12) for value in values)))


def rectangular_dp(
    prior_y1: float,
    source_projection_points: Sequence[Sequence[Tuple[float, float]]],
    costs: Array,
    abstention_cost: float,
) -> Tuple[Tuple[float, str, Dict[int, float], Tuple[float, float, float]], Dict]:
    """
    Fully rectangular outer approximation.

    The state retains the projected belief set but discards dependence between
    the belief and the source model that generated it.
    """

    n_sources = len(costs)
    channel_sets = [
        [make_channel(alpha, beta) for alpha, beta in source_points]
        for source_points in source_projection_points
    ]

    memo: Dict = {}

    def recurse(belief_set: Tuple[float, ...], remaining: Tuple[int, ...]):
        belief_set = _deduplicate_beliefs(belief_set)
        key = (belief_set, remaining)

        if key in memo:
            return memo[key]

        beliefs = np.asarray(belief_set, dtype=float)
        if not len(beliefs):
            memo[key] = (0.0, "abstain", {}, (0.0, 0.0, float(abstention_cost)))
            return memo[key]

        answer_0_risk = float(np.max(beliefs))
        answer_1_risk = float(np.max(1.0 - beliefs))
        risks = [answer_0_risk, answer_1_risk, abstention_cost]
        actions = ["answer_0", "answer_1", "abstain"]

        stopping_value = float(min(risks))
        stopping_action = actions[int(np.argmin(risks))]

        best_value = stopping_value
        best_action = stopping_action
        continuation_values: Dict[int, float] = {}

        for source_index in remaining:
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )

            child_belief_sets: Dict[int, Tuple[float, ...]] = {}

            for evidence in (0, 1):
                posteriors: List[float] = []

                for belief in beliefs:
                    for channel in channel_sets[source_index]:
                        denominator = (
                            (1.0 - belief) * channel[0, evidence]
                            + belief * channel[1, evidence]
                        )

                        if denominator > 0.0:
                            posterior = (
                                belief * channel[1, evidence] / denominator
                            )
                            posteriors.append(float(posterior))

                child_belief_sets[evidence] = _deduplicate_beliefs(posteriors)

            child_values = {
                evidence: recurse(
                    child_belief_sets[evidence], next_remaining
                )[0]
                for evidence in (0, 1)
            }

            worst_expectation = -np.inf

            for belief in beliefs:
                for channel in channel_sets[source_index]:
                    expectation = 0.0

                    for evidence in (0, 1):
                        evidence_probability = (
                            (1.0 - belief) * channel[0, evidence]
                            + belief * channel[1, evidence]
                        )
                        expectation += (
                            evidence_probability * child_values[evidence]
                        )

                    worst_expectation = max(
                        worst_expectation, expectation
                    )

            continuation_value = float(
                costs[source_index] + worst_expectation
            )
            continuation_values[source_index] = continuation_value

            if continuation_value < best_value - 1e-12:
                best_value = continuation_value
                best_action = f"query_{source_index}"

        memo[key] = (
            best_value,
            best_action,
            continuation_values,
            (answer_0_risk, answer_1_risk, abstention_cost),
        )
        return memo[key]

    root = recurse((prior_y1,), tuple(range(n_sources)))
    return root, memo


def static_best_response(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
    adversary_distribution: Array,
):
    """
    Best deterministic policy against a distribution over fixed source models.

    The recursion returns a per-scenario risk vector. At the root, this vector
    is the expected total loss of the policy under each fixed scenario.
    """

    n_scenarios, n_sources, _, _ = scenario_channels.shape
    memo: Dict = {}

    def likelihood_matrix(history: Tuple[Tuple[int, int], ...]) -> Array:
        likelihood = np.tile(
            np.array([1.0 - prior_y1, prior_y1], dtype=float),
            (n_scenarios, 1),
        )

        for source_index, evidence in history:
            likelihood *= scenario_channels[:, source_index, :, evidence]

        return likelihood

    def recurse(
        history: Tuple[Tuple[int, int], ...],
        remaining: Tuple[int, ...],
    ):
        key = (history, remaining)

        if key in memo:
            return memo[key]

        likelihood = likelihood_matrix(history)
        reach_probability = likelihood.sum(axis=1)

        candidates = [
            ("answer_0", likelihood[:, 1].copy(), None),
            ("answer_1", likelihood[:, 0].copy(), None),
            (
                "abstain",
                abstention_cost * reach_probability.copy(),
                None,
            ),
        ]

        for source_index in remaining:
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )
            child_zero = recurse(
                history + ((source_index, 0),), next_remaining
            )
            child_one = recurse(
                history + ((source_index, 1),), next_remaining
            )

            risk_vector = (
                costs[source_index] * reach_probability
                + child_zero[1]
                + child_one[1]
            )

            candidates.append(
                (
                    f"query_{source_index}",
                    risk_vector,
                    (child_zero[2], child_one[2]),
                )
            )

        weighted_values = [
            float(np.dot(adversary_distribution, candidate[1]))
            for candidate in candidates
        ]

        selected_index = int(np.argmin(weighted_values))
        action, risk_vector, children = candidates[selected_index]
        policy_tree = (action, children)

        memo[key] = (
            weighted_values[selected_index],
            risk_vector,
            policy_tree,
        )
        return memo[key]

    root = recurse(tuple(), tuple(range(n_sources)))
    return root, memo


def _solve_restricted_adversary(risk_matrix: Array):
    """
    Solve max_mu min_p <risk_p, mu> for the current policy columns.
    """

    n_policies, n_scenarios = risk_matrix.shape

    # Variables: mu_1,...,mu_S,v. Minimize -v.
    objective = np.zeros(n_scenarios + 1)
    objective[-1] = -1.0

    inequality_matrix = np.zeros((n_policies, n_scenarios + 1))
    inequality_matrix[:, :n_scenarios] = -risk_matrix
    inequality_matrix[:, -1] = 1.0
    inequality_rhs = np.zeros(n_policies)

    equality_matrix = np.zeros((1, n_scenarios + 1))
    equality_matrix[0, :n_scenarios] = 1.0
    equality_rhs = np.array([1.0])

    bounds = [(0.0, None)] * n_scenarios + [(None, None)]

    result = linprog(
        objective,
        A_ub=inequality_matrix,
        b_ub=inequality_rhs,
        A_eq=equality_matrix,
        b_eq=equality_rhs,
        bounds=bounds,
        method="highs",
    )

    if not result.success:
        raise RuntimeError(
            f"Restricted static game failed: {result.message}"
        )

    adversary_distribution = result.x[:n_scenarios]
    game_value = float(result.x[-1])

    return game_value, adversary_distribution


def static_double_oracle(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
    tolerance: float = 1e-9,
    max_iterations: int = 500,
):
    """
    Solve the exact finite-scenario randomized Static-ROSE game.

    A double-oracle method avoids enumerating all deterministic policy trees.
    The minimizer's best response is obtained by dynamic programming.
    """

    n_scenarios = scenario_channels.shape[0]
    adversary_distribution = np.ones(n_scenarios) / n_scenarios

    first_response, _ = static_best_response(
        prior_y1,
        scenario_channels,
        costs,
        abstention_cost,
        adversary_distribution,
    )

    policy_risk_vectors = [first_response[1]]
    policy_trees = [first_response[2]]
    observed_vectors = {tuple(np.round(first_response[1], 12))}
    convergence_history = []

    for iteration in range(max_iterations):
        risk_matrix = np.vstack(policy_risk_vectors)

        upper_bound, adversary_distribution = (
            _solve_restricted_adversary(risk_matrix)
        )

        best_response, _ = static_best_response(
            prior_y1,
            scenario_channels,
            costs,
            abstention_cost,
            adversary_distribution,
        )

        lower_bound = float(
            np.dot(adversary_distribution, best_response[1])
        )
        gap = float(upper_bound - lower_bound)

        convergence_history.append(
            {
                "iteration": iteration,
                "upper_bound": upper_bound,
                "lower_bound": lower_bound,
                "gap": gap,
                "n_policy_columns": len(policy_risk_vectors),
            }
        )

        if gap <= tolerance:
            return (
                upper_bound,
                adversary_distribution,
                np.vstack(policy_risk_vectors),
                policy_trees,
                convergence_history,
            )

        vector_key = tuple(np.round(best_response[1], 12))

        if vector_key in observed_vectors:
            # Numerical stalling at machine precision.
            return (
                upper_bound,
                adversary_distribution,
                np.vstack(policy_risk_vectors),
                policy_trees,
                convergence_history,
            )

        observed_vectors.add(vector_key)
        policy_risk_vectors.append(best_response[1])
        policy_trees.append(best_response[2])

    raise RuntimeError(
        "Static double-oracle solver reached its iteration limit."
    )


def _robust_stop_from_likelihood(
    likelihood: Array,
    abstention_cost: float,
) -> float:
    reach_probability = likelihood.sum(axis=1)
    active = reach_probability > 0.0

    if not np.any(active):
        return 0.0

    posterior_y1 = (
        likelihood[active, 1] / reach_probability[active]
    )

    return float(
        min(
            np.max(posterior_y1),
            np.max(1.0 - posterior_y1),
            abstention_cost,
        )
    )


def root_acquisition_certificates(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
):
    """
    Compute root-level perfect-information lower bounds and query-then-stop
    upper bounds for all source actions.
    """

    import pandas as pd

    n_scenarios, n_sources, _, _ = scenario_channels.shape
    root_likelihood = np.tile(
        np.array([1.0 - prior_y1, prior_y1], dtype=float),
        (n_scenarios, 1),
    )

    hcr_root, _ = hcr_dp(
        prior_y1, scenario_channels, costs, abstention_cost
    )
    stopping_value = float(min(hcr_root[3]))

    rows = []

    for source_index in range(n_sources):
        remaining = tuple(
            index for index in range(n_sources) if index != source_index
        )

        child_stopping_values = {}
        for evidence in (0, 1):
            next_likelihood = (
                root_likelihood
                * scenario_channels[:, source_index, :, evidence]
            )
            child_stopping_values[evidence] = (
                _robust_stop_from_likelihood(
                    next_likelihood, abstention_cost
                )
            )

        lower_scenario_values = []
        upper_scenario_values = []

        for scenario_index in range(n_scenarios):
            perfect_information_expectation = 0.0
            query_then_stop_expectation = 0.0

            for evidence in (0, 1):
                next_omega = (
                    root_likelihood[scenario_index]
                    * scenario_channels[
                        scenario_index, source_index, :, evidence
                    ]
                )
                evidence_probability = float(next_omega.sum())

                if evidence_probability > 0.0:
                    perfect_information_expectation += (
                        evidence_probability
                        * nominal_dp_from_state(
                            next_omega,
                            scenario_channels[scenario_index],
                            costs,
                            abstention_cost,
                            remaining,
                        )
                    )
                    query_then_stop_expectation += (
                        evidence_probability
                        * child_stopping_values[evidence]
                    )

            lower_scenario_values.append(
                costs[source_index] + perfect_information_expectation
            )
            upper_scenario_values.append(
                costs[source_index] + query_then_stop_expectation
            )

        lower_bound = float(max(lower_scenario_values))
        upper_bound = float(max(upper_scenario_values))
        exact_hcr_value = float(
            hcr_root[2][source_index]
        )

        rows.append(
            {
                "source_index": source_index,
                "LB": lower_bound,
                "C_HCR": exact_hcr_value,
                "UB": upper_bound,
                "stop_value": stopping_value,
                "bound_valid": (
                    lower_bound
                    <= exact_hcr_value + 1e-10
                    and exact_hcr_value
                    <= upper_bound + 1e-10
                ),
                "rejection_certificate": (
                    lower_bound >= stopping_value - 1e-12
                ),
                "no_harm_certificate": (
                    upper_bound < stopping_value - 1e-12
                ),
            }
        )

    return pd.DataFrame(rows)


def posterior_dilation_unit_test() -> Dict[str, float]:
    """
    Reproduce the analytical cost-free retrieval dilation example.
    """

    prior_y1 = 0.9
    abstention_cost = 0.4
    costs = np.array([0.0])

    channel_a = make_channel(alpha=0.9, beta=0.1)
    channel_b = make_channel(alpha=0.1, beta=0.9)

    scenarios = np.stack(
        [
            np.stack([channel_a]),
            np.stack([channel_b]),
        ]
    )

    root, _ = hcr_dp(
        prior_y1, scenarios, costs, abstention_cost
    )

    initial_stopping_value = float(min(root[3]))
    continuation_value = float(root[2][0])
    robust_value_of_evidence = (
        initial_stopping_value - continuation_value
    )

    expected = {
        "initial_stopping_value": 0.1,
        "post_retrieval_hcr_value": 0.4,
        "RVoE_HCR": -0.3,
    }

    actual = {
        "initial_stopping_value": initial_stopping_value,
        "post_retrieval_hcr_value": continuation_value,
        "RVoE_HCR": robust_value_of_evidence,
    }

    for key, expected_value in expected.items():
        if not np.isclose(actual[key], expected_value, atol=1e-10):
            raise AssertionError(
                f"Dilation unit test failed for {key}: "
                f"expected {expected_value}, obtained {actual[key]}."
            )

    return actual




from typing import Any
import math
SEED = 20261002
N_SOURCES = 3
ROBUST_LEVELS = [0.05,0.10,0.20]
ADVERSARIAL_SOURCE_FRACTION = 1/3
STATIC_SOLVER_TOL = 1e-9
def generate_nominal_problem(
    instance_id: int,
) -> dict[str, Any]:
    rng = np.random.default_rng(
        np.random.SeedSequence(
            [SEED, instance_id]
        )
    )

    sources: list[SourceSpec] = []

    for source_index in range(N_SOURCES):
        while True:
            alpha = float(rng.uniform(0.72, 0.93))
            beta = float(rng.uniform(0.07, 0.28))

            if alpha - beta >= 0.38:
                break

        cost = float(rng.uniform(0.008, 0.08))

        sources.append(
            SourceSpec(
                alpha=alpha,
                beta=beta,
                cost=cost,
                name=f"S{source_index + 1}",
            )
        )

    prior_y1 = float(rng.uniform(0.10, 0.90))
    abstention_cost = float(rng.uniform(0.20, 0.45))

    nominal_channels = np.stack(
        [
            make_channel(
                source.alpha,
                source.beta,
            )
            for source in sources
        ]
    )

    costs = np.array(
        [source.cost for source in sources],
        dtype=float,
    )

    return {
        "sources": sources,
        "prior_y1": prior_y1,
        "abstention_cost": abstention_cost,
        "nominal_channels": nominal_channels,
        "costs": costs,
    }

def build_robust_scenario_family(
    sources: list[SourceSpec],
) -> tuple[
    np.ndarray,
    list[list[tuple[float, float]]],
    list[str],
]:
    """
    Construct a finite coupled ambiguity family around the nominal model.

    It contains:
    - coherent reliability and positive-rate shifts;
    - source-specific degradation;
    - source-specific positive-rate bias.

    Rect-ROSE later combines source projections arbitrarily.
    """
    regimes: list[
        tuple[np.ndarray, np.ndarray, str]
    ] = []

    n_sources = len(sources)
    nominal_alpha = np.array(
        [source.alpha for source in sources],
        dtype=float,
    )
    nominal_beta = np.array(
        [source.beta for source in sources],
        dtype=float,
    )

    regimes.append(
        (
            nominal_alpha.copy(),
            nominal_beta.copy(),
            "nominal",
        )
    )

    for level in ROBUST_LEVELS:
        coherent_shifts = [
            (-level, +level, "degraded"),
            (+level, -level, "optimistic"),
            (-level, -level, "low_rate"),
            (+level, +level, "high_rate"),
        ]

        for delta_alpha, delta_beta, name in coherent_shifts:
            regimes.append(
                (
                    np.clip(
                        nominal_alpha + delta_alpha,
                        1e-6,
                        1.0 - 1e-6,
                    ),
                    np.clip(
                        nominal_beta + delta_beta,
                        1e-6,
                        1.0 - 1e-6,
                    ),
                    f"{name}_{level:.2f}",
                )
            )

    for source_index in range(n_sources):
        for level in [0.10, 0.20]:
            alpha = nominal_alpha.copy()
            beta = nominal_beta.copy()

            alpha[source_index] = np.clip(
                alpha[source_index] - level,
                1e-6,
                1.0 - 1e-6,
            )
            beta[source_index] = np.clip(
                beta[source_index] + level,
                1e-6,
                1.0 - 1e-6,
            )

            regimes.append(
                (
                    alpha,
                    beta,
                    (
                        f"source_{source_index + 1}_"
                        f"degraded_{level:.2f}"
                    ),
                )
            )

        alpha = nominal_alpha.copy()
        beta = nominal_beta.copy()

        alpha[source_index] = np.clip(
            alpha[source_index] + 0.10,
            1e-6,
            1.0 - 1e-6,
        )
        beta[source_index] = np.clip(
            beta[source_index] + 0.20,
            1e-6,
            1.0 - 1e-6,
        )

        regimes.append(
            (
                alpha,
                beta,
                f"source_{source_index + 1}_positive_bias",
            )
        )

    scenario_channels: list[np.ndarray] = []
    scenario_names: list[str] = []
    seen = set()

    for alpha_vector, beta_vector, name in regimes:
        channels = np.stack(
            [
                make_channel(
                    float(alpha),
                    float(beta),
                )
                for alpha, beta in zip(
                    alpha_vector,
                    beta_vector,
                )
            ]
        )

        key = tuple(
            np.round(channels.ravel(), 12)
        )

        if key in seen:
            continue

        seen.add(key)
        scenario_channels.append(channels)
        scenario_names.append(name)

    scenarios = np.stack(scenario_channels)

    source_projection_points: list[
        list[tuple[float, float]]
    ] = []

    for source_index in range(n_sources):
        points: list[tuple[float, float]] = []

        for scenario_index in range(
            scenarios.shape[0]
        ):
            channel = scenarios[
                scenario_index,
                source_index,
            ]
            point = (
                float(channel[1, 1]),
                float(channel[0, 1]),
            )

            if not any(
                np.allclose(point, existing)
                for existing in points
            ):
                points.append(point)

        source_projection_points.append(points)

    return (
        scenarios,
        source_projection_points,
        scenario_names,
    )

def mix_channel_collections(
    nominal_channels: np.ndarray,
    target_channels: np.ndarray,
    xi: float,
) -> np.ndarray:
    result = (
        (1.0 - xi) * nominal_channels
        + xi * target_channels
    )

    # Mixtures of valid row-stochastic channels remain valid, but
    # normalization is applied for numerical safety.
    result = result / result.sum(
        axis=-1,
        keepdims=True,
    )
    return result

def target_channels_for_shift(
    nominal_channels: np.ndarray,
    costs: np.ndarray,
    shift_type: str,
) -> np.ndarray:
    n_sources = nominal_channels.shape[0]
    target = nominal_channels.copy()

    if shift_type == "degradation":
        for source_index in range(n_sources):
            target[source_index] = make_channel(
                alpha=0.55,
                beta=0.45,
            )

    elif shift_type == "asymmetric_bias":
        for source_index in range(n_sources):
            target[source_index] = make_channel(
                alpha=0.95,
                beta=0.78,
            )

    elif shift_type == "adversarial_source":
        n_attacked = max(
            1,
            int(
                math.ceil(
                    ADVERSARIAL_SOURCE_FRACTION
                    * n_sources
                )
            ),
        )

        # Attack the cheapest sources, which are likely to be selected
        # by a nominal risk-cost policy.
        attacked_indices = np.argsort(
            costs
        )[:n_attacked]

        for source_index in attacked_indices:
            target[source_index] = make_channel(
                alpha=0.10,
                beta=0.90,
            )

    elif shift_type == "common_mode_failure":
        for source_index in range(n_sources):
            target[source_index] = make_channel(
                alpha=0.52,
                beta=0.48,
            )

    else:
        raise ValueError(
            f"Unknown shift type: {shift_type}"
        )

    return target

def true_regimes_for_shift(
    nominal_channels: np.ndarray,
    costs: np.ndarray,
    shift_type: str,
    xi: float,
) -> tuple[list[np.ndarray], np.ndarray]:
    """
    Return the fixed true regimes and their episode-level probabilities.

    The first three shift types produce one fixed shifted channel model.

    common_mode_failure uses an episode-level latent regime:
    - nominal regime with probability 1-xi;
    - joint failure regime with probability xi.

    This creates cross-source dependence marginally across episodes.
    """
    target = target_channels_for_shift(
        nominal_channels,
        costs,
        shift_type,
    )

    if shift_type == "common_mode_failure":
        if xi <= 0.0:
            return [nominal_channels], np.array([1.0])

        if xi >= 1.0:
            return [target], np.array([1.0])

        return (
            [nominal_channels, target],
            np.array([1.0 - xi, xi]),
        )

    shifted = mix_channel_collections(
        nominal_channels,
        target,
        xi,
    )
    return [shifted], np.array([1.0])

def solve_restricted_policy_mixture(
    policy_risk_matrix: np.ndarray,
) -> tuple[np.ndarray, float]:
    """
    Solve the minimizer's restricted primal game.

    policy_risk_matrix[p,s] is the risk of deterministic policy p
    under ambiguity scenario s.
    """
    n_policies, n_scenarios = (
        policy_risk_matrix.shape
    )

    # Variables are policy weights w_1,...,w_P and scalar t.
    objective = np.zeros(n_policies + 1)
    objective[-1] = 1.0

    inequality_matrix = np.zeros(
        (n_scenarios, n_policies + 1)
    )
    inequality_matrix[:, :n_policies] = (
        policy_risk_matrix.T
    )
    inequality_matrix[:, -1] = -1.0
    inequality_rhs = np.zeros(n_scenarios)

    equality_matrix = np.zeros(
        (1, n_policies + 1)
    )
    equality_matrix[0, :n_policies] = 1.0
    equality_rhs = np.array([1.0])

    bounds = (
        [(0.0, None)] * n_policies
        + [(None, None)]
    )

    result = linprog(
        objective,
        A_ub=inequality_matrix,
        b_ub=inequality_rhs,
        A_eq=equality_matrix,
        b_eq=equality_rhs,
        bounds=bounds,
        method="highs",
    )

    if not result.success:
        raise RuntimeError(
            "Static policy-mixture LP failed: "
            f"{result.message}"
        )

    weights = np.asarray(result.x[:n_policies], dtype=float).copy()
    if not np.all(np.isfinite(weights)) or np.min(weights) < -1e-8:
        raise RuntimeError("Policy-mixture LP returned invalid policy weights.")
    # Repair only negative solver roundoff; retain every positive policy mass.
    weights = np.maximum(weights, 0.0)
    mass = float(weights.sum())
    if mass <= 0.0 or abs(mass - 1.0) > 1e-8:
        raise RuntimeError("Policy-mixture weights do not sum to one within tolerance.")
    weights /= mass
    value = float(np.max(policy_risk_matrix.T @ weights))
    if abs(value - float(result.x[-1])) > 1e-8:
        raise RuntimeError("Implemented policy mixture disagrees with the LP objective.")
    return weights, value

def build_static_rose_policy(
    prior_y1: float,
    robust_scenarios: np.ndarray,
    costs: np.ndarray,
    abstention_cost: float,
) -> dict[str, Any]:
    (
        static_value,
        adversary_distribution,
        policy_risk_matrix,
        policy_trees,
        convergence_history,
    ) = static_double_oracle(
        prior_y1,
        robust_scenarios,
        costs,
        abstention_cost,
        tolerance=STATIC_SOLVER_TOL,
    )

    policy_weights, restricted_value = (
        solve_restricted_policy_mixture(
            policy_risk_matrix
        )
    )

    if not np.isclose(
        static_value,
        restricted_value,
        atol=1e-7,
    ):
        raise AssertionError(
            "Static primal and dual values disagree: "
            f"{static_value} vs {restricted_value}."
        )

    return {
        "value": static_value,
        "policy_weights": policy_weights,
        "policy_trees": policy_trees,
        "adversary_distribution": (
            adversary_distribution
        ),
        "convergence_history": (
            convergence_history
        ),
    }

def rectangular_child_belief_set(
    belief_set: tuple[float, ...],
    source_channels: list[np.ndarray],
    evidence: int,
) -> tuple[float, ...]:
    posteriors: list[float] = []

    for belief in belief_set:
        for channel in source_channels:
            denominator = (
                (1.0 - belief)
                * channel[0, evidence]
                + belief
                * channel[1, evidence]
            )

            if denominator > 0.0:
                posterior = (
                    belief
                    * channel[1, evidence]
                    / denominator
                )
                posteriors.append(
                    float(posterior)
                )

    return _deduplicate_beliefs(posteriors)


In [ ]:
# ROSE-RAG Experiment 3B
# Coupled source-quality regimes with asymmetric retrieval costs
#
# This file is the experiment body.  A builder combines it with the original
# ROSE-RAG mathematical core to create a self-contained .py script and a
# self-contained Jupyter notebook.

from pathlib import Path
import json
import itertools
import platform
import time

import numpy as np
import pandas as pd



# Frozen, prospective protocol


PROTOCOL = {
    "protocol_version": "experiment3b-coupled-regime-confirmatory-v1",
    "seed": 20261008,
    "diagnostic_instance_ids": [19000, 19001, 19002],
    "final_test_id_start": 20000,
    "final_n_test": 100,
    "all_priors": [0.55, 0.60],
    "all_abstention_costs": [0.20, 0.25],
    "operational_priors": [0.55, 0.60],
    "operational_abstention_costs": [0.20, 0.25],
    "scenario_names": [
        "stable_confirmation",
        "source3_failure",
        "source1_degraded_source2_compensating",
    ],
    "scenario_weights": [1.0 / 3.0, 1.0 / 3.0, 1.0 / 3.0],
    "abstention_cost_is_loss": True,
    "bootstrap_repetitions": 5000,
    "bootstrap_seed_offset": 701,
    "primary_endpoint": (
        "confirmatory per-instance mean risk of HCRROSE versus StaticROSE, "
        "averaged over the three latent regimes and the prespecified "
        "candidate prior/cost grid"
    ),
    "secondary_endpoint": (
        "per-instance worst-regime risk averaged over operational priors "
        "and operational abstention costs"
    ),
    "primary_comparator": "StaticROSE",
    "hypothesis": (
        "On fresh instances from the same prespecified coupled-regime "
        "mechanism, HCRROSE has lower mean operational risk than "
        "StaticROSE in the candidate regime; all other comparisons are "
        "secondary."
    ),
    "candidate_regime_selected_post hoc": True,
    "no_test_tuning": True,
}

RUN_MODE = "final"  # change to "final" only after diagnostic succeeds
RESUME = True
OUTPUT_ROOT = Path("experiment3b_confirmatory_outputs")

METHODS = [
    "InternalOnly",
    "RetrieveAll",
    "FixedTop2",
    "NominalOneQuery",
    "NominalSequential",
    "RobustOneQuery",
    "StaticROSE",
    "HCRROSE",
    "RectROSE",
    "OracleRegime",
]

OUTCOMES = np.array(list(itertools.product([0, 1], repeat=3)), dtype=int)
METRICS = ["risk", "wrong", "abstain", "coverage", "cost", "queries"]



# Exact policy-tree and risk-evaluation utilities


def fixed_tree(prior_y1, channels, abstention_cost, order):
    """Build a deterministic fixed-order policy tree."""

    def recurse(position, weights):
        if position == len(order):
            posterior = float(weights[1] / weights.sum())
            action = stop_value_precise(posterior, abstention_cost)[1]
            return (action, None)

        source_index = int(order[position])
        children = tuple(
            recurse(position + 1, weights * channels[source_index, :, evidence])
            for evidence in (0, 1)
        )
        return (f"query_{source_index}", children)

    return recurse(0, np.array([1.0 - prior_y1, prior_y1], dtype=float))


def tree_from_hcr_memo(memo, n_sources=3):
    def recurse(history, remaining):
        action = memo[(history, remaining)][1]
        if not action.startswith("query_"):
            return (action, None)
        source_index = int(action.split("_")[1])
        next_remaining = tuple(j for j in remaining if j != source_index)
        children = tuple(
            recurse(history + ((source_index, evidence),), next_remaining)
            for evidence in (0, 1)
        )
        return (action, children)

    return recurse(tuple(), tuple(range(n_sources)))


def tree_from_rect_memo(prior_y1, source_projection_points, memo, n_sources=3):
    channel_sets = [
        [make_channel(alpha, beta) for alpha, beta in points]
        for points in source_projection_points
    ]

    def recurse(belief_set, remaining):
        belief_set = _deduplicate_beliefs(belief_set)
        action = memo[(belief_set, remaining)][1]
        if not action.startswith("query_"):
            return (action, None)
        source_index = int(action.split("_")[1])
        next_remaining = tuple(j for j in remaining if j != source_index)
        children = tuple(
            recurse(
                rectangular_child_belief_set(
                    belief_set, channel_sets[source_index], evidence
                ),
                next_remaining,
            )
            for evidence in (0, 1)
        )
        return (action, children)

    return recurse((prior_y1,), tuple(range(n_sources)))


def tree_tensor(tree, costs, abstention_cost):
    """Convert a policy tree into exact loss components.

    Output shape is (Y, source-outcome, metric), where metric is
    [risk, wrong, abstain, coverage, cost, queries].
    """

    tensor = np.zeros((2, len(OUTCOMES), len(METRICS)), dtype=float)

    for outcome_index, evidence_vector in enumerate(OUTCOMES):
        node = tree
        total_cost = 0.0
        n_queries = 0.0

        while node[0].startswith("query_"):
            source_index = int(node[0].split("_")[1])
            total_cost += float(costs[source_index])
            n_queries += 1.0
            node = node[1][int(evidence_vector[source_index])]

        action = node[0]
        is_abstain = float(action == "abstain")

        for y in (0, 1):
            is_wrong = float(
                (not is_abstain) and action != f"answer_{y}"
            )
            tensor[y, outcome_index] = [
                is_wrong + abstention_cost * is_abstain + total_cost,
                is_wrong,
                is_abstain,
                1.0 - is_abstain,
                total_cost,
                n_queries,
            ]

    return tensor


def scenario_distribution(prior_y1, channels):
    """Exact joint distribution of Y and three binary source outcomes."""

    channels = np.asarray(channels, dtype=float)
    if channels.ndim == 3:
        channels = channels[None, ...]

    probabilities = np.ones((len(channels), 2, len(OUTCOMES)), dtype=float)
    for source_index in range(3):
        probabilities *= np.stack(
            [
                channels[:, source_index, :, int(evidence)]
                for evidence in OUTCOMES[:, source_index]
            ],
            axis=-1,
        )

    probabilities *= np.array([1.0 - prior_y1, prior_y1])[None, :, None]
    if not np.allclose(probabilities.sum(axis=(1, 2)), 1.0, atol=1e-12):
        raise AssertionError("joint probability mass is not normalized")
    return probabilities


def evaluate_tensor(tensor, prior_y1, channels):
    probabilities = scenario_distribution(prior_y1, channels)
    return np.einsum("syo,yok->sk", probabilities, tensor)


def robust_one_query_tree(prior_y1, scenario_channels, costs, abstention_cost):
    actions = ("answer_0", "answer_1", "abstain")
    candidates = [(stop_value_precise(prior_y1, abstention_cost)[1], None)]
    for source_index in range(len(costs)):
        for action_zero, action_one in itertools.product(actions, repeat=2):
            candidates.append(
                (
                    f"query_{source_index}",
                    ((action_zero, None), (action_one, None)),
                )
            )

    robust_values = []
    for candidate in candidates:
        tensor = tree_tensor(candidate, costs, abstention_cost)
        robust_values.append(
            float(evaluate_tensor(tensor, prior_y1, scenario_channels)[:, 0].max())
        )

    return candidates[int(np.argmin(robust_values))], float(min(robust_values))


def nominal_one_query_tree(prior_y1, nominal_channels, costs, abstention_cost):
    candidates = [
        (stop_value_precise(prior_y1, abstention_cost)[1], None)
    ]
    for source_index in range(len(costs)):
        candidates.append(
            fixed_tree(
                prior_y1,
                nominal_channels,
                abstention_cost,
                (source_index,),
            )
        )

    values = []
    for candidate in candidates:
        tensor = tree_tensor(candidate, costs, abstention_cost)
        values.append(
            float(evaluate_tensor(tensor, prior_y1, nominal_channels)[0, 0])
        )

    return candidates[int(np.argmin(values))]


def nominal_tree(prior_y1, nominal_channels, costs, abstention_cost):
    """Build the exact nominal sequential-DP policy tree."""

    root, memo = nominal_dp(
        prior_y1,
        nominal_channels,
        costs,
        abstention_cost,
    )

    def recurse(omega, remaining):
        key = (
            tuple(float(value) for value in np.asarray(omega, dtype=float)),
            remaining,
        )
        if key not in memo:
            raise KeyError(
                "Nominal DP state missing while reconstructing policy tree: "
                f"remaining={remaining}, omega={omega}"
            )

        action = memo[key][1]
        if not action.startswith("query_"):
            return (action, None)

        source_index = int(action.split("_")[1])
        next_remaining = tuple(
            index for index in remaining if index != source_index
        )
        children = tuple(
            recurse(
                np.asarray(omega, dtype=float)
                * nominal_channels[source_index, :, evidence],
                next_remaining,
            )
            for evidence in (0, 1)
        )
        return (action, children)

    root_omega = np.array([1.0 - prior_y1, prior_y1], dtype=float)
    tree = recurse(root_omega, tuple(range(len(costs))))
    return root, tree


def build_policy_set(problem, prior_y1, abstention_cost):
    nominal = problem["nominal_channels"]
    robust = problem["robust_scenarios"]
    projection_points = problem["projection_points"]
    costs = problem["costs"]

    policies = {}

    policies["InternalOnly"] = (
        stop_value_precise(prior_y1, abstention_cost)[1],
        None,
    )

    one_query_trees = [
        fixed_tree(prior_y1, nominal, abstention_cost, (i,))
        for i in range(3)
    ]
    nominal_one_query_risks = [
        float(
            evaluate_tensor(
                tree_tensor(tree, costs, abstention_cost),
                prior_y1,
                nominal,
            )[0, 0]
        )
        for tree in one_query_trees
    ]
    nominal_order = tuple(np.argsort(nominal_one_query_risks, kind="stable"))

    policies["RetrieveAll"] = fixed_tree(
        prior_y1, nominal, abstention_cost, nominal_order
    )
    policies["FixedTop2"] = fixed_tree(
        prior_y1, nominal, abstention_cost, nominal_order[:2]
    )
    policies["NominalOneQuery"] = nominal_one_query_tree(
        prior_y1, nominal, costs, abstention_cost
    )
    _, policies["NominalSequential"] = nominal_tree(
        prior_y1, nominal, costs, abstention_cost
    )
    policies["RobustOneQuery"], robust_one_query_value = robust_one_query_tree(
        prior_y1, robust, costs, abstention_cost
    )

    hcr_root, hcr_memo = hcr_dp(
        prior_y1, robust, costs, abstention_cost
    )
    policies["HCRROSE"] = tree_from_hcr_memo(hcr_memo, n_sources=3)

    rect_root, rect_memo = rectangular_dp(
        prior_y1, projection_points, costs, abstention_cost
    )
    policies["RectROSE"] = tree_from_rect_memo(
        prior_y1, projection_points, rect_memo, n_sources=3
    )

    static_solution = build_static_rose_policy(
        prior_y1, robust, costs, abstention_cost
    )
    static_tensor = sum(
        weight * tree_tensor(tree, costs, abstention_cost)
        for weight, tree in zip(
            static_solution["policy_weights"],
            static_solution["policy_trees"],
        )
    )

    audit = {
        "nominal_one_query_risks": nominal_one_query_risks,
        "nominal_order": list(map(int, nominal_order)),
        "robust_one_query_value": robust_one_query_value,
        "static_value": float(static_solution["value"]),
        "static_gap": float(
            static_solution["convergence_history"][-1]["gap"]
        ),
        "hcr_value": float(hcr_root[0]),
        "rect_value": float(rect_root[0]),
        "static_policy_weights": static_solution["policy_weights"].tolist(),
        "hcr_root_action": hcr_root[1],
        "rect_root_action": rect_root[1],
        "hcr_tree": repr(policies["HCRROSE"]),
        "rect_tree": repr(policies["RectROSE"]),
    }

    return policies, static_tensor, audit



# Structured coupled-regime generator


def _jitter_pair(rng, alpha, beta, scale=0.012):
    alpha = float(np.clip(alpha + rng.normal(0.0, scale), 0.02, 0.98))
    beta = float(np.clip(beta + rng.normal(0.0, scale), 0.02, 0.98))
    return alpha, beta


def _channel_from_pair(pair):
    return make_channel(float(pair[0]), float(pair[1]))


def generate_structured_instance(instance_id):
    """Generate one independent instance from the prespecified mechanism.

    The three latent regimes have operational interpretations:
      A: cheap screening and expensive confirmation are reliable;
      B: a third source has a failure/adversarial episode;
      C: the cheap first source degrades while the expensive source remains
         highly informative.

    The nominal model is deliberately a validation-time approximation and is
    not tuned from the test outcomes.
    """

    rng = np.random.default_rng(
        np.random.SeedSequence([PROTOCOL["seed"], int(instance_id)])
    )

    nominal_pairs = [
        _jitter_pair(rng, 0.82, 0.15),
        _jitter_pair(rng, 0.85, 0.30),
        _jitter_pair(rng, 0.60, 0.28),
    ]

    scenario_templates = [
        [
            (0.90, 0.10),
            (0.75, 0.15),
            (0.60, 0.40),
        ],
        [
            (0.90, 0.20),
            (0.75, 0.15),
            (0.30, 0.70),
        ],
        [
            (0.60, 0.05),
            (0.95, 0.15),
            (0.80, 0.15),
        ],
    ]

    robust_scenarios = np.stack(
        [
            np.stack(
                [
                    _channel_from_pair(_jitter_pair(rng, alpha, beta))
                    for alpha, beta in template
                ]
            )
            for template in scenario_templates
        ]
    )

    # Source 1 and source 3 are cheap screening channels; source 2 is a
    # relatively expensive confirmation channel.
    costs = np.array(
        [
            rng.uniform(0.001, 0.005),
            rng.uniform(0.050, 0.080),
            rng.uniform(0.001, 0.005),
        ],
        dtype=float,
    )

    projection_points = []
    for source_index in range(3):
        points = []
        for scenario in robust_scenarios:
            point = (
                float(scenario[source_index, 1, 1]),
                float(scenario[source_index, 0, 1]),
            )
            if not any(np.allclose(point, previous) for previous in points):
                points.append(point)
        projection_points.append(points)

    return {
        "nominal_channels": np.stack(
            [_channel_from_pair(pair) for pair in nominal_pairs]
        ),
        "robust_scenarios": robust_scenarios,
        "scenario_names": list(PROTOCOL["scenario_names"]),
        "scenario_weights": np.array(PROTOCOL["scenario_weights"], dtype=float),
        "projection_points": projection_points,
        "costs": costs,
        "nominal_pairs": nominal_pairs,
    }



# Row generation, checkpointing, summaries, and paired bootstrap


def _metric_rows(
    instance_id,
    prior_y1,
    abstention_cost,
    problem,
    policies,
    static_tensor,
):
    rows = []
    robust = problem["robust_scenarios"]
    scenario_names = problem["scenario_names"]
    scenario_weights = problem["scenario_weights"]
    costs = problem["costs"]

    tensors = {
        name: tree_tensor(tree, costs, abstention_cost)
        for name, tree in policies.items()
    }
    tensors["StaticROSE"] = static_tensor

    for method, tensor in tensors.items():
        values = evaluate_tensor(tensor, prior_y1, robust)
        for scenario_index, metrics in enumerate(values):
            rows.append(
                {
                    "instance_id": int(instance_id),
                    "prior": float(prior_y1),
                    "abstention_cost": float(abstention_cost),
                    "scenario_index": int(scenario_index),
                    "scenario": scenario_names[scenario_index],
                    "scenario_weight": float(scenario_weights[scenario_index]),
                    "method": method,
                    **dict(zip(METRICS, metrics)),
                }
            )

    # OracleRegime is a reference only: it may use the true latent regime.
    oracle_rows = []
    for scenario_index, scenario in enumerate(robust):
        _, oracle_tree = nominal_tree(
            prior_y1,
            scenario,
            costs,
            abstention_cost,
        )
        oracle_tensor = tree_tensor(oracle_tree, costs, abstention_cost)
        oracle_values = evaluate_tensor(
            oracle_tensor,
            prior_y1,
            scenario[None, ...],
        )[0]
        oracle_rows.append(
            {
                "instance_id": int(instance_id),
                "prior": float(prior_y1),
                "abstention_cost": float(abstention_cost),
                "scenario_index": int(scenario_index),
                "scenario": scenario_names[scenario_index],
                "scenario_weight": float(scenario_weights[scenario_index]),
                "method": "OracleRegime",
                **dict(zip(METRICS, oracle_values)),
            }
        )

    return rows + oracle_rows


def _checkpoint_valid(path, expected_rows, instance_id):
    if not path.exists():
        return False
    try:
        saved = pd.read_csv(path)
        required = {
            "instance_id",
            "prior",
            "abstention_cost",
            "scenario",
            "method",
            "risk",
        }
        return (
            len(saved) == expected_rows
            and required.issubset(saved.columns)
            and saved["instance_id"].eq(instance_id).all()
            and not saved.duplicated(
                [
                    "instance_id",
                    "prior",
                    "abstention_cost",
                    "scenario_index",
                    "method",
                ]
            ).any()
        )
    except (OSError, EOFError, ValueError, pd.errors.EmptyDataError):
        return False


def _instance_ids(run_mode):
    if run_mode == "diagnostic":
        return list(PROTOCOL["diagnostic_instance_ids"])
    if run_mode == "final":
        start = PROTOCOL["final_test_id_start"]
        stop = start + PROTOCOL["final_n_test"]
        return list(range(start, stop))
    raise ValueError("RUN_MODE must be 'diagnostic' or 'final'")


def run_experiment(output_dir=None, run_mode=None, resume=True):
    run_mode = run_mode or RUN_MODE
    output = Path(output_dir or OUTPUT_ROOT / run_mode)
    output.mkdir(parents=True, exist_ok=True)
    checkpoint_dir = output / "checkpoints"
    checkpoint_dir.mkdir(exist_ok=True)

    protocol = dict(PROTOCOL)
    protocol["run_mode"] = run_mode
    protocol["instance_ids"] = _instance_ids(run_mode)
    (output / "protocol.json").write_text(
        json.dumps(protocol, indent=2), encoding="utf-8"
    )

    expected_rows = (
        len(PROTOCOL["all_priors"])
        * len(PROTOCOL["all_abstention_costs"])
        * len(PROTOCOL["scenario_names"])
        * len(METHODS)
    )

    audit_rows = []
    start = time.perf_counter()

    for position, instance_id in enumerate(protocol["instance_ids"], start=1):
        checkpoint = checkpoint_dir / f"{instance_id}.csv.gz"
        audit_checkpoint = checkpoint_dir / f"{instance_id}_audit.json"

        if resume and _checkpoint_valid(checkpoint, expected_rows, instance_id):
            continue

        problem = generate_structured_instance(instance_id)
        rows = []
        local_audit = {
            "instance_id": int(instance_id),
            "costs": problem["costs"].tolist(),
            "nominal_pairs": [list(pair) for pair in problem["nominal_pairs"]],
            "scenario_names": problem["scenario_names"],
            "scenario_pairs": [
                [
                    [
                        float(scenario[source_index, 1, 1]),
                        float(scenario[source_index, 0, 1]),
                    ]
                    for source_index in range(3)
                ]
                for scenario in problem["robust_scenarios"]
            ],
            "policy_audits": [],
        }

        for prior_y1 in PROTOCOL["all_priors"]:
            for abstention_cost in PROTOCOL["all_abstention_costs"]:
                policies, static_tensor, policy_audit = build_policy_set(
                    problem,
                    prior_y1,
                    abstention_cost,
                )
                rows.extend(
                    _metric_rows(
                        instance_id,
                        prior_y1,
                        abstention_cost,
                        problem,
                        policies,
                        static_tensor,
                    )
                )
                local_audit["policy_audits"].append(
                    {
                        "prior": float(prior_y1),
                        "abstention_cost": float(abstention_cost),
                        **policy_audit,
                    }
                )

        frame = pd.DataFrame(rows)
        temporary = checkpoint.with_name(checkpoint.name + ".tmp")
        frame.to_csv(temporary, index=False, compression="gzip")
        temporary.replace(checkpoint)
        audit_temporary = audit_checkpoint.with_name(
            audit_checkpoint.name + ".tmp"
        )
        audit_temporary.write_text(
            json.dumps(local_audit, indent=2), encoding="utf-8"
        )
        audit_temporary.replace(audit_checkpoint)
        audit_rows.append(local_audit)

        print(
            f"Completed {position}/{len(protocol['instance_ids'])} "
            f"(instance {instance_id}); "
            f"elapsed {time.perf_counter() - start:.1f}s",
            flush=True,
        )

    paths = [
        checkpoint_dir / f"{instance_id}.csv.gz"
        for instance_id in protocol["instance_ids"]
    ]
    frame = pd.concat([pd.read_csv(path) for path in paths], ignore_index=True)
    expected_total = expected_rows * len(protocol["instance_ids"])
    if len(frame) != expected_total:
        raise AssertionError(
            f"expected {expected_total} rows, obtained {len(frame)}"
        )

    frame.to_csv(
        output / "instance_results.csv.gz",
        index=False,
        compression="gzip",
    )
    (output / "solver_audit.json").write_text(
        json.dumps(audit_rows, indent=2), encoding="utf-8"
    )
    (output / "run_environment.json").write_text(
        json.dumps(
            {
                "python": platform.python_version(),
                "numpy": np.__version__,
                "pandas": pd.__version__,
                "run_mode": run_mode,
                "n_rows": int(len(frame)),
                "elapsed_seconds": float(time.perf_counter() - start),
                "note": (
                    "The final test set is generated from fixed seeds and "
                    "the protocol is not tuned from test outcomes."
                ),
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    summarize_results(frame, output)
    return frame


def summarize_results(frame, output_dir):
    output = Path(output_dir)
    operational = frame[
        frame["prior"].isin(PROTOCOL["operational_priors"])
        & frame["abstention_cost"].isin(
            PROTOCOL["operational_abstention_costs"]
        )
    ].copy()

    all_summary = (
        frame.groupby(
            ["prior", "abstention_cost", "scenario", "method"],
            as_index=False,
        )[METRICS]
        .mean()
    )
    all_summary.to_csv(output / "risk_by_prior_cost_scenario.csv", index=False)

    operational.to_csv(output / "operational_rows.csv", index=False)

    per_instance = (
        operational.groupby(["instance_id", "method"], as_index=False)
        .agg(
            mean_risk=("risk", "mean"),
            worst_regime_risk=("risk", "max"),
            mean_wrong=("wrong", "mean"),
            mean_abstain=("abstain", "mean"),
            mean_cost=("cost", "mean"),
            mean_queries=("queries", "mean"),
        )
    )
    per_instance.to_csv(output / "operational_per_instance.csv", index=False)

    ranking = (
        per_instance.groupby("method", as_index=False)
        .agg(
            mean_risk=("mean_risk", "mean"),
            mean_worst_regime_risk=("worst_regime_risk", "mean"),
            mean_wrong=("mean_wrong", "mean"),
            mean_abstain=("mean_abstain", "mean"),
            mean_cost=("mean_cost", "mean"),
            mean_queries=("mean_queries", "mean"),
        )
        .sort_values("mean_risk")
    )
    ranking.to_csv(output / "operational_ranking.csv", index=False)

    bootstrap_comparisons(per_instance, output)
    write_plot_if_available(ranking, output)


def bootstrap_comparisons(per_instance, output_dir):
    output = Path(output_dir)
    wide = per_instance.pivot(
        index="instance_id",
        columns="method",
        values=["mean_risk", "worst_regime_risk"],
    )
    available_methods = set(wide.columns.get_level_values(1))
    comparators = [
        method
        for method in METHODS
        if method not in {"HCRROSE", "OracleRegime"}
        and method in available_methods
    ]

    rows = []
    rng = np.random.default_rng(
        PROTOCOL["seed"] + PROTOCOL["bootstrap_seed_offset"]
    )
    n_instances = len(wide.index)
    if n_instances == 0:
        return

    for endpoint in ["mean_risk", "worst_regime_risk"]:
        delta_matrix = np.column_stack(
            [
                wide[(endpoint, "HCRROSE")].to_numpy()
                - wide[(endpoint, comparator)].to_numpy()
                for comparator in comparators
            ]
        )
        observed = delta_matrix.mean(axis=0)
        indices = rng.integers(
            0,
            n_instances,
            size=(PROTOCOL["bootstrap_repetitions"], n_instances),
        )
        bootstrap_values = delta_matrix[indices].mean(axis=1)
        simultaneous_radius = float(
            np.quantile(
                np.max(
                    np.abs(bootstrap_values - observed[None, :]),
                    axis=1,
                ),
                0.95,
            )
        )

        for index, comparator in enumerate(comparators):
            rows.append(
                {
                    "endpoint": endpoint,
                    "comparator": comparator,
                    "n_instances": int(n_instances),
                    "HCR_minus_comparator": float(observed[index]),
                    "pointwise_low": float(
                        np.quantile(bootstrap_values[:, index], 0.025)
                    ),
                    "pointwise_high": float(
                        np.quantile(bootstrap_values[:, index], 0.975)
                    ),
                    "simultaneous_low": float(
                        observed[index] - simultaneous_radius
                    ),
                    "simultaneous_high": float(
                        observed[index] + simultaneous_radius
                    ),
                }
            )

    pd.DataFrame(rows).to_csv(
        output / "paired_bootstrap_operational.csv",
        index=False,
    )


def write_plot_if_available(ranking, output_dir):
    import matplotlib.pyplot as plt
    from pathlib import Path

    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)

    display_labels = {
        "OracleRegime": "Oracle\n(regime)",
        "HCRROSE": "HCR-ROSE",
        "StaticROSE": "Static-ROSE",
        "NominalSequential": "Nominal\nSequential",
        "FixedTop2": "Fixed\nTop-2",
        "RectROSE": "Rect-ROSE",
        "NominalOneQuery": "Nominal\nOne-Query",
        "RetrieveAll": "Retrieve\nAll",
        "RobustOneQuery": "Robust\nOne-Query",
        "InternalOnly": "Internal\nOnly",
    }

    plot_data = ranking.sort_values("mean_risk").reset_index(drop=True)
    labels = [
        display_labels.get(method, method)
        for method in plot_data["method"]
    ]

    positions = list(range(len(plot_data)))

    fig, ax = plt.subplots(figsize=(10.5, 5.6))

    ax.bar(
        positions,
        plot_data["mean_risk"],
        color="#4472C4",
        width=0.72
    )

    ax.set_xticks(positions)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel("Mean operational risk", fontsize=11)
    ax.set_title(
        "Experiment 3B: coupled regimes and asymmetric retrieval cost",
        fontsize=12,
        pad=10
    )

    ax.tick_params(axis="y", labelsize=9)
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)
    ax.margins(x=0.02)

    fig.subplots_adjust(
        left=0.10,
        right=0.99,
        bottom=0.19,
        top=0.88
    )

    fig.savefig(
        output / "figure_operational_ranking_confirmatory.pdf",
        bbox_inches="tight",
        pad_inches=0.04
    )

    fig.savefig(
        output / "figure_operational_ranking_confirmatory.svg",
        bbox_inches="tight",
        pad_inches=0.04
    )

    plt.close(fig)

def recover_results(output_dir=None, run_mode=None):
    """Reload an already completed run without recomputing policies."""

    run_mode = run_mode or RUN_MODE
    output = Path(output_dir or OUTPUT_ROOT / run_mode)
    frame = pd.read_csv(output / "instance_results.csv.gz")
    summarize_results(frame, output)
    return frame



# Experiment execution


if __name__ == "__main__":
    results = run_experiment(
        output_dir=OUTPUT_ROOT / RUN_MODE,
        run_mode=RUN_MODE,
        resume=RESUME,
    )
    print("\nOperational ranking:")
    print(
        pd.read_csv(OUTPUT_ROOT / RUN_MODE / "operational_ranking.csv")
        .round(6)
        .to_string(index=False)
    )
    print("\nPaired bootstrap comparisons:")
    print(
        pd.read_csv(OUTPUT_ROOT / RUN_MODE / "paired_bootstrap_operational.csv")
        .round(6)
        .to_string(index=False)
    )


## Recover saved results

`recover_results(OUTPUT_DIR)` rebuilds summaries from completed output files without repeating the confirmatory calculation.
